In [1]:
import pandas as pd
import os
from Bio import SeqIO, AlignIO
import pysam
# from Bio.Align.Applications import PrankCommandline
# import subprocess
from collections import Counter
import statistics
import re
import csv
from Bio.SeqIO import FastaIO
import Levenshtein

from multiprocessing import Pool, cpu_count
from tqdm import tqdm
import numpy as np

In [2]:
"""
call mutations
"""
def dictBam(bamfile, outDict):
    """
    i
    """
    with pysam.AlignmentFile(bamfile, "rb", check_sq=False) as infile:
        with open(outDict, "w") as outfile:
#         dic ={}
            for read in infile.fetch(until_eof=True):
                dic = {}
#                 if not read.has_tag("MD") or (query_seq := read.query_sequence) is None or read.is_secondary:  ##fixed on 10/18/2022
#                     return None
                if read.has_tag("MD") and (query_seq := read.query_sequence) is not None and not read.is_secondary:
                    readDict = callMutation(read)
                    dic[read.qname] = readDict
                    outfile.write(f"{read.qname}\t{readDict[0]}\t{readDict[1]}\t{readDict[2]}\t{readDict[3]}\n")
def callMutation(read):
    """
    get mutations and indels from a read which has MD tag in sam/bam
    """

    mut_type = []
    mut_qual= []
    indel_type = []
    indel_qual =[]

    ref_index = read.reference_start-1
    query_index = -1
    I_index = [0]
    I_qual = []
    D_index = []
    D_qual = []

    
    
    for query_pos, ref_pos, ref_base in read.get_aligned_pairs(with_seq=True): #matches_only=True, # if True, no None on either side

        if query_pos is None: # Deletion
            ref_index += 1
            D_index.append(ref_index)
            D_qual.append(read.query_qualities[query_index])
#                     if D_index[-1]-D_index[-2] >1:
#                         indel_type.append(str(ref_index)+"D")
#                         indel_qual.append(read.query_qualities[query_index])

        elif ref_pos is None: # Insertion
            query_index +=1
            I_index.append(ref_index)
            I_qual.append(read.query_qualities[query_index])
            if I_index[-1]-I_index[-2] >1:
                indel_type.append(str(ref_index)+"I")
                indel_qual.append(read.query_qualities[query_pos])

        elif ref_base.islower():
            ref_index += 1
            query_index +=1
            mut_type.append(str(ref_pos)+ref_base.upper()+read.query_sequence[query_pos])
            mut_qual.append(read.query_qualities[query_pos])

        else:
            ref_index += 1
            query_index +=1
    D = concatDel(D_index, D_qual,"D")
    indel_type = indel_type + D[0]
    indel_qual = indel_qual + D[1]
    return(mut_type, mut_qual, indel_type, indel_qual)

def concatDel(A,B,type="D"):
    A.append(-1)
    l = len(A)
    i = 0
    j = 0
    inds = []
    OUT= []
    QUAL = []
    while i < l-2:
        while j < l-1:
            if not (A[j+1] - A[j] - 1):
                j += 1
            else:
                i = j + 1
                j = i
                inds.append(j)
    inds.insert(0, 0)
    for i in range(len(inds)-1):
        if not inds[i+1] - inds[i] - 1:
            OUT.append(str(A[inds[i]])+type)
            QUAL.append(B[inds[i]])
        else:
            OUT.append('{}-{}{}'.format(A[inds[i]], A[inds[i+1]-1],type))
            qual = B[inds[i]:inds[i+1]]
            qual_avg = int(sum(qual)/len(qual))
            QUAL.append(qual_avg)
    return(OUT, QUAL)


def ccSeq(sub, frequency_threshold, quality_threshold):
    """
    find cluster consensus mutations in each uc group
    """
    num = len(sub)
    ccMuts = []
    
    muts =sub["mut_type"].to_numpy()
    muts = removeBracket(muts, qual = False)
    mut_quals =sub["mut_qual"].to_numpy()
    mut_quals = removeBracket(mut_quals, qual = True)
    indels =sub["indel_type"].to_numpy()
    indels = removeBracket(indels, qual = False)
    indel_quals =sub["indel_qual"].to_numpy()
    indel_quals = removeBracket(indel_quals, qual = True)
      
    fM = filterMuts(muts, mut_quals, num, frequency_threshold, quality_threshold)
    fID = filterMuts(indels, indel_quals, num, frequency_threshold, quality_threshold)
    ccMuts.append(fM)
    ccMuts.append(fID)
    ccMuts = flatList(ccMuts)
    return(ccMuts)

def removeBracket(s,qual=True):
    """
    ori: ["['2048TC']" '[]' "['1283CT', '1529GC']"]
    transformed: ['2048TC','1283CT','1529GC']
    """
    su = []
    if s is not None:
        for s_sub in s:
            s_sub = s_sub.replace("[","")
            s_sub = s_sub.replace("]","")
            s_sub = s_sub.replace("'","")
            s_sub = s_sub.split(",")
            for s_sub_sub in s_sub:
                if s_sub_sub != '':
                    s_sub_sub = s_sub_sub.replace(" ","")
                    if qual:
                        su.append(int(s_sub_sub))
                    else:
                        su.append(str(s_sub_sub))
    return(su)
    
def filterMuts(muts, mut_quals, num, frequency_threshold = 0.5, quality_threshold = 50):
    """
    filter mutations/indels with frequency >= frequency_threshold
    """
    cs = [ele for ele, cnt in Counter(muts).items() if cnt >= num*frequency_threshold]
    csMuts = []
    if len(cs) >0:
        for mutType in cs:
            quals = []
            for b,q in zip(muts, mut_quals):
                if b == mutType:
                    quals.append(q)
            if statistics.mean(quals) >= quality_threshold:
                csMuts.append(mutType)
    return(csMuts)


def flatList(Alist):
    flat_list = []
    for sublist in Alist:
        for item in sublist:
            flat_list.append(item)        
    return(flat_list)


##-----------------------------------------------------------------------
"""
create allele matrix
"""
def parseDel(D,maxLen=200):
    for s in D:
        pos = re.findall("(\d+)",s)
        if len(pos) ==2:
            if int(pos[1])-int(pos[0]) > maxLen:
                return("LongDels")
def parseSubs(subs, ref):
    pos=[re.findall(r'[0-9]+',s)[0] for s in subs]
    types=[re.findall(r'[A-Z]+',s)[0] for s in subs]
    pos=[re.findall(r'[0-9]+',s)[0] for s in subs]
    types=[re.findall(r'[A-Z]+',s)[0] for s in subs]
    pos_m=[]
    for i in pos:
        pos_m.append(int(i))
    type_m=[]
    for i in types:
        type_m.append(i)
    s1=[0]*476
    s2=ref
    for i in range(len(pos_m)):
        pointer = pos_m[i] #-1
        s1[pointer]=1
        s2 = s2[:pointer] + type_m[i][1] + s2[pointer + 1:]
    s0=''.join(map(str,s1))
    return(s0, s2)   ## s0: binary; s2:character sequence

def createMatrix(infile, OUTDIR, my_sample, REF):
    ref = open(REF,"r").readlines()[1].replace("\n", "")
    #out_fasta=OUTDIR+my_sample+".fa"           ## output 1. fasta file
    out_allele=OUTDIR+my_sample+"_allele-table.txt"               ## output 2. binary file
    out_phy=OUTDIR+my_sample+"_temp.phy"                   ## output .phy file for iqtree reconstruction

    with open(infile,"r") as f_in, open(out_allele, "w") as f_allele:#, open(out_phy, "w") as f_phy, open (out_fasta, "w") as f_fasta
        f_in_lines = f_in.readlines()[1:]
        numCell = len(f_in_lines)+1
        #f_phy.write(str("ref")+" "+''.join(map(str,[0]*476))+"\n")
        #f_fasta.write(">"+str("ref")+"\n"+ref+"\n")
        for line in f_in_lines:
            muts=line.replace("\n","").split("\t")
            muts
            cell_name=muts[0]
            staticBarcode_name=muts[1]
            numUMI=muts[2]
            readCount=muts[3]
            sample=muts[7]
            muts.pop(0)
            muts.pop(0)
            muts.pop(0)
            muts.pop(0)
            muts.pop(0)
            muts = muts[0].split(",")
            types=[re.findall(r'[A-Z]+',s) for s in muts]
            l = len(types)
            subs = []
            #### deletions ############
            D = []
            for i in range(l):
                if types[i]==["D"]:
                    D.append(muts[i])
            D_long = parseDel(D,maxLen=450)
            #### insertions ###########
            I = []
            for i in range(l):
                if types[i]==["I"]:
                    I.append(muts[i])

            ### substitutions ########

            if D_long is None or "LongDels" not in D_long:
                subs = [ele for ele in muts if ele not in D and ele not in I]
                if len(subs)==1 and subs[0]=="":
                    matrix_binary = ''.join(map(str,[0]*476))
                    matrix_character = ref
                else:    
                    matrixALL = parseSubs(subs, ref)
                    matrix_binary = matrixALL[0]
                    matrix_character = matrixALL[1].replace("\n", "")

                f_allele.write(cell_name+"\t"+staticBarcode_name+"\t"+sample+"\t"+numUMI+"\t"+readCount+"\t"+matrix_character+"\t"+matrix_binary+"\n") ## add number of UMIs in each static barcode!!
                #f_phy.write(cell_name+"_"+staticBarcode_name+" "+matrix_binary+"\n")
                #f_fasta.write(">"+cell_name+"_"+staticBarcode_name+"\n"+matrix_character+"\n")
def makeConsensus(group):
    cellBC,sub = group
    
    counts=sub.groupby(['intBC','umi']).size().reset_index(name='counts')
    counts=counts[counts['counts']>=numRead]
    counts_umi=counts.groupby(['intBC']).size().reset_index(name='counts')
    counts_umi=counts_umi[counts_umi['counts']>=numUMI]

    
    for my_intBC in counts_umi.intBC.unique():
        sub_df=sub[(sub['intBC']==my_intBC) & (sub['umi'].isin(counts['umi']))]
        
        ## collapse each umi group as a consensus
        umis=sub_df.umi.unique()
        intBCMuts = []
        for my_umi in umis:
            sub3=sub_df.loc[sub_df["umi"]==my_umi]
            ssMuts = ccSeq(sub3, mutFreq, quality_threshold)
            intBCMuts.append(ssMuts)
                
        ## make each static barcode as consensus!!!
        Mutation_list = Counter(flatList(intBCMuts))
        csMutation=[]
        for mut, freq in Mutation_list.items():
            if freq >= len(umis)*mutFreq:
                csMutation.append(mut)
        ## make each static barcode as consensus end
        s=len(csMutation)   ## s: number of mutations
        if s >0:
            return {'cellBC': cellBC[0], 'intBC': my_intBC, 'umiCount': len(umis),'readCount': sub_df.shape[0],'numMut': s,'mut':','.join([i for i in csMutation])}
        elif s==0:
            return {'cellBC': cellBC[0], 'intBC': my_intBC, 'umiCount': len(umis),'readCount': sub_df.shape[0],'numMut': s,'mut':''}
            
def reverse_complement(sequence):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement[base] for base in reversed(sequence))


In [3]:
if __name__ == "__main__":
    """
    get mutations
    """
    samples=['C4007-C4011']
    FASTQDIR="/syn2/zhaolian/3.JiLab/results/3.PacBio/1.SMALTout/3.C4007-C4011/02.fastq/"
    BAMDIR="/syn2/zhaolian/3.JiLab/results/3.PacBio/1.SMALTout/3.C4007-C4011/03.bam_split/"
    OUTDIR="/syn2/zhaolian/3.JiLab/results/3.PacBio/1.SMALTout/3.C4007-C4011/04.mutation_alleleTable/"
    REF="/syn2/zhaolian/3.JiLab/reference/TargetOnly476.fasta"
    if not os.path.exists(OUTDIR):
        os.mkdir(OUTDIR)
    
    my_sample=samples[0]
    print("Calculating "+my_sample+"...")
    
    bamfile = BAMDIR+my_sample+".bam"
    umi_table = FASTQDIR+my_sample+".txt"
    index=pd.read_csv("/syn2/zhaolian/3.JiLab/results/3.PacBio/index3_"+my_sample+".txt",delimiter="\t")
    numRead=1               ## minimum number of reads in each UMI
    numUMI=1                 ## minimum number of UMIs in each static barcode
    mutFreq=0.6              ## export the mutation if mutation frequency >= mutFreq 
    quality_threshold = 10   ## mapping quality
    outfile=OUTDIR+my_sample+"_mutations_"+str(numRead)+"_"+str(numUMI)+".txt" ## final output file 2
    outDict = OUTDIR+my_sample+".mutations" ## intermediate output file 1
    ###########################################
    ## get mutations file from bamfile, run only once !!!
    # dictBam(bamfile, outDict)    
    colnames =["qname","cellBC", "umi", "intBC","i7","i5"]
    umi_table = pd.read_csv(umi_table,delimiter="\t",header=None,usecols=[0,1,2,4,6,8],names=colnames)
    print("Number of raw reads (strand-specific) ", str(umi_table.shape[0]))
    ###########################################
    umi_table=umi_table[~umi_table['cellBC'].isna()]
    umi_table=umi_table[~umi_table['umi'].isna()]
    umi_table=umi_table[~umi_table['intBC'].isna()]
    # umi_table = umi_table.dropna()
    print("Number of reads after dropna", str(umi_table.shape[0]))
    ###########################################
    colnames =["qname","mut_type", "mut_qual", "indel_type", "indel_qual"]
    dictbam = pd.read_csv(outDict,delimiter="\t",header=None,names=colnames)
    dictbam.qname="@"+dictbam.qname
    umi_table=umi_table[~umi_table.duplicated(['qname'], keep=False)]
    dictbam=dictbam[~dictbam.duplicated(['qname'], keep=False)]
    cellBCbam=pd.merge(umi_table,dictbam, on="qname",how="inner")
    cellBCbam['cellBC'] = cellBCbam['cellBC'].apply(reverse_complement)    
    print("Number of raw reads with mutations", str(cellBCbam.shape[0]))
    ###########################################
    ## split samples based on index
    cellBCbam['i7_i5']=cellBCbam['i7']+'_'+cellBCbam['i5']
    del cellBCbam['i7']
    del cellBCbam['i5']
    index['i7_i5']=index['i7']+'_'+index['i5']
    index['mouse_sample']=index['mouse']+'_'+index['sample']
    del index['i7']
    del index['i5']
    cellBCbam=pd.merge(index,cellBCbam, on="i7_i5",how="right")
    dumpDF=cellBCbam[cellBCbam['mouse'].isna()]
    cellBCbam=cellBCbam[~cellBCbam['mouse'].isna()]
    print("Number of reads split by index ", str(cellBCbam.shape[0]))
    print("Number of reads excluded by index ", str(dumpDF.shape[0]))
    ###########################################
    ## split based on cellBC present in scRNAseq
    # bc= pd.read_csv('/syn2/zhaolian/3.JiLab/results/scRNAseq_clean_metadata_all_bulkName.txt',sep='\t', header=0)
    bc= pd.read_csv('/syn2/zhaolian/3.JiLab/results/2.scRNAseq/3.integrated/scdataDF_merged_metadata.txt',usecols=[2,0,1],sep='\t', header=0)
    bc['sampleID']=bc['mouse']+'_'+bc['sampleID']  #### mouse, individual sample
    batch= pd.read_csv('/syn2/zhaolian/3.JiLab/results/2.scRNAseq/cellNum_batch.txt',sep='\t', header=0,usecols=[0,2],names=["sampleID","mouse_sample"])
    bc=pd.merge(bc,batch, on="sampleID",how="inner")
    # bc['re'] = bc['cellBC'].apply(reverse_complement)    
    bc=bc[bc['mouse_sample'].isin(index['mouse_sample'])]
    # del bc["sampleID"]
    del dumpDF['mouse']
    del dumpDF['sample']
    del dumpDF['mouse_sample']
    # dumpDF=dumpDF[dumpDF['cellBC'].isin(bc['cellBC'])]
    dumpDF=pd.merge(bc,dumpDF, on="cellBC",how="inner")
    print("Number of reads dumped reads recovered by cellBC ", str(dumpDF.shape[0]))
    ###########################################
    ## final reads with mouse and sample info
    print(len(cellBCbam[cellBCbam["cellBC"].isin(bc["cellBC"])]))
    ## remove chimera
    bc['index']=bc["mouse_sample"]+"_"+bc["cellBC"]
    del bc['mouse_sample']
    del bc['cellBC']
    del bc["mouse"]
    ######################################################
    ######################################################
    ## cellBC correction
    cellBCbamCorrected=pd.DataFrame()
    for my_mouse in cellBCbam['mouse'].unique():
        sub=cellBCbam[cellBCbam['mouse']==my_mouse]
        whitelist="/syn2/zhaolian/3.JiLab/results/2.scRNAseq/3.integrated/whitelist-cellBC-scRNA20240923-"+my_mouse+".txt"
        if isinstance(whitelist, list):
            whitelist_set = set(whitelist)
        else:
            with open(whitelist, "r") as f:
                whitelist_set = set(
                    line.strip() for line in f if not line.isspace()
                )
        whitelist = list(whitelist_set)
        
        unique_cellbcs = list(sub["cellBC"].unique())
        corrections = {cellbc: cellbc for cellbc in whitelist_set}
        print(f"{my_mouse}: {len(unique_cellbcs)} cellBCs detected.")
        cellbc_dist_thresh=1
        for cellbc in tqdm(unique_cellbcs, desc="Correcting cellBCs to whitelist"):
            min_distance = np.inf
            min_wls = []
            if cellbc not in whitelist_set:
                for wl_cellbc in whitelist:
                    distance = Levenshtein.distance(cellbc, wl_cellbc)
                    if distance < min_distance:
                        min_distance = distance
                        min_wls = [wl_cellbc]
                    elif distance == min_distance:
                        min_wls.append(wl_cellbc)
        
            # Correct only if there is one matching whitelist. Discard if there
            # are multiple possible corrections.
            if len(min_wls) == 1 and min_distance <= cellbc_dist_thresh:
                corrections[cellbc] = min_wls[0]
        
        sub["cellBC"] = sub["cellBC"].map(corrections)
        unique_cellbcs = list(sub["cellBC"].unique())
        print(f"{my_mouse}: {len(unique_cellbcs)} cellBCs after correction.")
        cellBCbamCorrected=pd.concat([cellBCbamCorrected,sub],ignore_index=True)
    ######################################################
    ######################################################
    cellBCbamCorrected['index']=cellBCbamCorrected['mouse_sample']+"_"+cellBCbamCorrected['cellBC']
    cellBCbam_merged=pd.merge(cellBCbamCorrected,bc, on="index",how="inner")
    print(len(cellBCbam_merged))
    dumpDF["sample"] = [i.split("_")[1] for i in dumpDF["mouse_sample"]]
    del cellBCbam_merged['index']
    print("Number of reads: split by index and assigned to sampleID ", str(cellBCbam_merged.shape[0]))
    df=pd.concat([cellBCbam_merged, dumpDF],ignore_index=True)
    print("Number of assigned reads ", str(df.shape[0]))
    df.to_csv("/syn2/zhaolian/3.JiLab/results/3.PacBio/2.alleleTable/"+my_sample+"_assigned_reads.txt", sep='\t',index=False,header=True)    


Calculating C4007-C4011...


/tmp/ipykernel_275336/12111951.py:29: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  umi_table = pd.read_csv(umi_table,delimiter="\t",header=None,usecols=[0,1,2,4,6,8],names=colnames)


Number of raw reads (strand-specific)  4410929
Number of reads after dropna 3705933
Number of raw reads with mutations 3685478
Number of reads split by index  3357573
Number of reads excluded by index  327905
Number of reads dumped reads recovered by cellBC  201275
2074079
C4007: 120819 cellBCs detected.


Correcting cellBCs to whitelist: 100%|██████████| 120819/120819 [51:22<00:00, 39.19it/s] 
/tmp/ipykernel_275336/12111951.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub["cellBC"] = sub["cellBC"].map(corrections)


C4007: 32919 cellBCs after correction.
C4011: 135091 cellBCs detected.


Correcting cellBCs to whitelist: 100%|██████████| 135091/135091 [37:17<00:00, 60.38it/s]
/tmp/ipykernel_275336/12111951.py:121: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sub["cellBC"] = sub["cellBC"].map(corrections)


C4011: 32241 cellBCs after correction.
2067799
Number of reads: split by index and assigned to sampleID  2067799
Number of assigned reads  2269074
